# recommendations_total 과 게임 흥행의 관계 분석

`recommendations_total`(Steam 추천 수)이 흥행 지표(`total_reviews`)와 어떤 관계가 있는지 분석합니다.

- **`recommendations_total`**: Steam 상세 페이지의 유저 추천 수
- **`total_reviews`**: 흥행 주 지표 (Steam 공식 집계 리뷰 수)
- **`owners_lower`**: 흥행 보조 지표 (Steam Spy 추정 판매량 하한)

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

pd.set_option('display.max_columns', None)

In [14]:
# 데이터 로드 및 병합
sample_df = pd.read_csv('../../../data/processed/steam_indie_9692.csv')
details_df = pd.read_csv('../../../data/processed/steam_indie_details.csv')

df = pd.merge(
    sample_df,
    details_df[['appid', 'recommendations_total']],
    on='appid',
    how='left'
)

# recommendations_total 결측 제거
df_valid = df.dropna(subset=['recommendations_total']).copy()

print(f"전체 high 계층 게임 수: {len(df)}")
print(f"recommendations_total 유효 게임 수: {len(df_valid)}")
print(f"결측 게임 수: {len(df) - len(df_valid)}")
print(f"\n계층별 유효 게임 수:")
df_valid[['name', 'total_reviews', 'owners_lower', 'recommendations_total']].head()

전체 high 계층 게임 수: 9692
recommendations_total 유효 게임 수: 91
결측 게임 수: 9601

계층별 유효 게임 수:


,name,total_reviews,owners_lower,recommendations_total
2,CyberCorp,322,10000000,328.0
11,Balatro,153566,2000000,152600.0
23,MiSide,111087,1000000,117968.0
32,Necesse,17071,1000000,22876.0
45,DELTARUNE,2835,1000000,90730.0


In [15]:
# 기초 통계
print("=== recommendations_total 기초 통계 ===")
print(df_valid[['recommendations_total', 'total_reviews', 'owners_lower']].describe().round(1))

=== recommendations_total 기초 통계 ===
       recommendations_total  total_reviews  owners_lower
count                   91.0           91.0          91.0
mean                  6531.7         5315.7      268461.5
std                  22208.4        19897.3     1072259.2
min                    101.0           90.0           0.0
25%                    293.5          301.0       10000.0
50%                    555.0          524.0       50000.0
75%                   2944.5         2893.0      200000.0
max                 152600.0       153566.0    10000000.0


In [16]:
# Spearman 상관계수: recommendations_total vs 흥행 지표
corr_reviews, p_reviews = stats.spearmanr(df_valid['recommendations_total'], df_valid['total_reviews'])
corr_owners, p_owners = stats.spearmanr(df_valid['recommendations_total'], df_valid['owners_lower'])

print("=== recommendations_total vs 흥행 지표 Spearman 상관계수 ===")
print(f"vs total_reviews : r = {corr_reviews:.3f}, p = {p_reviews:.4f} {'(유의미) *' if p_reviews < 0.05 else '(비유의미)'}")
print(f"vs owners_lower  : r = {corr_owners:.3f}, p = {p_owners:.4f} {'(유의미) *' if p_owners < 0.05 else '(비유의미)'}")

=== recommendations_total vs 흥행 지표 Spearman 상관계수 ===
vs total_reviews : r = 0.967, p = 0.0000 (유의미) *
vs owners_lower  : r = 0.606, p = 0.0000 (유의미) *


In [17]:
# 산점도: recommendations_total vs total_reviews
fig = px.scatter(
    df_valid,
    x='recommendations_total',
    y='total_reviews',
    text='name',
    trendline='ols',
    title=f'recommendations_total vs total_reviews (Spearman r={corr_reviews:.3f}, p={p_reviews:.4f})',
    labels={
        'recommendations_total': '추천 수 (recommendations_total)',
        'total_reviews': '총 리뷰 수 (total_reviews)'
    },
    template='plotly_white'
)
fig.update_traces(textposition='top center')
fig.update_layout(height=600)
fig.show()

In [18]:
# 로그 변환 산점도 (우편향 분포 보정)
df_valid['log_rec'] = np.log1p(df_valid['recommendations_total'])
df_valid['log_reviews'] = np.log1p(df_valid['total_reviews'])

corr_log, p_log = stats.spearmanr(df_valid['log_rec'], df_valid['log_reviews'])

fig_log = px.scatter(
    df_valid,
    x='log_rec',
    y='log_reviews',
    text='name',
    trendline='ols',
    title=f'[로그 변환] recommendations_total vs total_reviews (r={corr_log:.3f}, p={p_log:.4f})',
    labels={
        'log_rec': 'log(recommendations_total)',
        'log_reviews': 'log(total_reviews)'
    },
    template='plotly_white'
)
fig_log.update_layout(height=600)
fig_log.show()

In [20]:
# 결론 요약
print("=" * 65)
print("recommendations_total 흥행 영향 분석 요약")
print("=" * 65)
print(f"\n[분석 대상] {len(df_valid)}개 게임 (high 계층, recommendations_total 유효)")
print(f"\n[상관계수]")
print(f"  vs total_reviews : r = {corr_reviews:.3f}, p = {p_reviews:.4f} {'*' if p_reviews < 0.05 else ''}")
print(f"  vs owners_lower  : r = {corr_owners:.3f}, p = {p_owners:.4f} {'*' if p_owners < 0.05 else ''}")
print(f"\n[해석]")
if corr_reviews > 0.7 and p_reviews < 0.05:
    print(f"  recommendations_total은 흥행 지표와 강한 양의 상관 → 흥행 지표로 활용 가능")
elif corr_reviews > 0.4 and p_reviews < 0.05:
    print(f"  recommendations_total은 흥행 지표와 중간 수준의 양의 상관")
    print(f"  → 단독 지표보다는 보조 지표로 활용 적합")
elif p_reviews >= 0.05:
    print(f"  통계적으로 유의미한 상관관계 없음 → 흥행 지표로 적합하지 않음")
else:
    print(f"  약한 상관관계 → 흥행 지표로 단독 사용 부적합")
print("=" * 65)

recommendations_total 흥행 영향 분석 요약

[분석 대상] 91개 게임 (high 계층, recommendations_total 유효)

[상관계수]
  vs total_reviews : r = 0.967, p = 0.0000 *
  vs owners_lower  : r = 0.606, p = 0.0000 *

[해석]
  recommendations_total은 흥행 지표와 강한 양의 상관 → 흥행 지표로 활용 가능
